In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer 

data = load_breast_cancer()

features = data.data
labels = data.target

df = pd.DataFrame(features, columns=data.feature_names)
df['labels'] = labels

print("Feature Shape: ",features.shape)
print("Label Shape: ", labels.shape)

unique, count = np.unique(labels, return_counts=True)

for label, count in zip(data.target_names, count):
    print(f"{label}: {count}")

# The dataset is somewhat imbalanced, with around 37.3% malignant, and 62.7% benign.

# Models can clock a high accuracy purely by guessing the majority (benign), leading the model
# to learn absolutely nothing about the datasets patterns, trends, etc..

Feature Shape:  (569, 30)
Label Shape:  (569,)
malignant: 212
benign: 357


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

xTrain, xTest, yTrain, yTest = train_test_split(features, labels, test_size=0.2, random_state=22, stratify=labels)

decisionTreeModel = DecisionTreeClassifier(criterion="entropy", random_state=221)
decisionTreeModel.fit(xTrain, yTrain)

trainingAccuracy = decisionTreeModel.score(xTrain, yTrain)
testingAccuracy = decisionTreeModel.score(xTest, yTest)

print(f"Training Accuracy: {trainingAccuracy}")
print(f"Testing Accuracy: {testingAccuracy}")

# Entropy measures the impurity of a certain node. A node with a 50/50 split has the maximum entropy,
# whilst something with a 100/0 split would have an entropy of 0.
# Splits within the decision tree means that the model picks a feature and tries to minimize its entropy.

# The decision tree doesn't have a max_depth, which is usually the paramter to utilize when there is overfitting.
# In this case, there is an atrocious amount of overfitting within the model.
# A small gap between training, and testing accuracy means that there is good generalization however.


Training Accuracy: 1.0
Testing Accuracy: 0.8596491228070176


In [ ]:
modifiedTreeModel = DecisionTreeClassifier(criterion="entropy", max_depth=4, min_samples_split=10, random_state=221)

modifiedTreeModel.fit(xTrain, yTrain)

mtmTrainAccuracy = modifiedTreeModel.score(xTrain, yTrain)
mtmTestAccuracy = modifiedTreeModel.score(xTest, yTest)
print(f"Training Accuracy (Modified Model) {mtmTrainAccuracy}")
print(f"Testing Accuracy (Modified Model): {mtmTestAccuracy}")

importance = modifiedTreeModel.feature_importances_
indices = np.argsort(importance)[::-1][:5]

print("\nTop 5 Most Important Features:")
for rank, i in enumerate(indices, 1):
    print(f"{rank}. {data.feature_names[i]}: {importance[i]}")

# Controlling the complexity of the model helps in controlling the growth of the tree,
# holding off on doing so until the leaf is pure.

# Feature importances tells us which features are the most impactful in reducing the entropy across every single split.
# This gives us the needed interpretability to identify which features are most telling in malignancy.

Training Accuracy (Modified Model) 0.9846153846153847
Testing Accuracy (Modified Model): 0.8596491228070176

Top 5 Most Important Features:
1. worst area: 0.6947768639075658
2. worst concave points: 0.16245975422778555
3. mean texture: 0.05161911531626712
4. worst concavity: 0.04806016541252051
5. mean smoothness: 0.029878779694166687


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(xTrain)
xTestScaled  = scaler.transform(xTest)

neuralNetwork = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', max_iter=500, random_state=221)
neuralNetwork.fit(xTrainScaled, yTrain)

trainingAccuracy = neuralNetwork.score(xTrainScaled, yTrain)
testAccuracy = neuralNetwork.score(xTestScaled, yTest)
print(f"Training Accuracy: {trainingAccuracy}")
print(f"Test Accuracy: {testAccuracy}")

# Feature scaling helps standardize the dataset, this is necessary because some models are generally
# sensitive towards the scaling of features inputted in. This can skew the model in a way that does not
# accurately represent the dataset.

# An epoch is a complete pass within the neural network. Training samples, and neurons are went over once
# in one epoch, performing calculations on measurements such as loss, and making predicts in the process.1

Training Accuracy: 0.9956043956043956
Test Accuracy: 0.9473684210526315


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

dtPredictions = modifiedTreeModel.predict(X_test)
nnPredictions = neuralNetwork.predict(xTestScaled)

dtConfusionMatrix = confusion_matrix(y_test, dtPredictions)
nnConfusionMatrix = confusion_matrix(y_test, nnPredictions)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(dtConfusionMatrix, display_labels=data.target_names).plot(ax=axes[0])
axes[0].set_title("Constrained Decision Tree")

ConfusionMatrixDisplay(nnConfusionMatrix, display_labels=data.target_names).plot(ax=axes[1])
axes[1].set_title("Neural Network")

plt.tight_layout()
plt.show()

# The neural network would be preferred. Even though the performance metrics state that they are almost even, neural networks are generally better
# with false negatives, which is a very grave thing to have within the context of the medical field.

# Decision trees are very interpretable, and we can show which features or thresholds contributed to how much in making a prediction.
# However, even with constraints being placed using its parameters, decision trees are sensitive to small changes within the dataset.

# Neural Networks are good with generalization as a result of epochs. The network is able to continuously optimize.
# However, they are a black box, meaning its hard to understand why it made a certain prediction, it just did.

In [3]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

(mXTrain, mYTrain), (mXTest, mYTest) = fashion_mnist.load_data()
 
mXTrain = mXTrain / 255.0 # Pixel normalization here, just dividing everything by the standard rgb limit 255
mXTest = mXTest  / 255.0
 
mXTrain = mXTrain.reshape(-1, 28, 28, 1)
mXTest = mXTest.reshape(-1, 28, 28, 1)
 
model = Sequential([
    Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
 
    Flatten(),
 
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
 
model.summary()
 
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
 
history = model.fit(
    mXTrain, mYTrain,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
 
testLoss, testAcc = model.evaluate(mXTest, mYTest, verbose=0)
print(f"\nTest Accuracy: {testAcc}")
print(f"Test Loss: {testLoss}")

# CNNs take into account the actual spatial structures of the images. Rather than treating each indiviual
# pixel as its own thing, CNNS look at smalle portions/regions of the image with filters, which helps bring down the 
# number of parameters. Being able to break things down into regions also helps in recognizing more subtle patterns within
# the images.

# The filters are learning things such as the color gradients, edges, etc.. in the low level. Deeper layers however
# combine all the low level features into larger patterns, things like soles, collars, sleeves, etc.. 

2026-03-23 12:45:17.864134: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/lib/python3.14/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

2026-03-23 12:45:20.194023: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


Epoch 1/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - accuracy: 0.8222 - loss: 0.4952 - val_accuracy: 0.8672 - val_loss: 0.3757
Epoch 2/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.8816 - loss: 0.3295 - val_accuracy: 0.8852 - val_loss: 0.3178
Epoch 3/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.8974 - loss: 0.2822 - val_accuracy: 0.8948 - val_loss: 0.2788
Epoch 4/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9087 - loss: 0.2492 - val_accuracy: 0.9020 - val_loss: 0.2713
Epoch 5/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9179 - loss: 0.2249 - val_accuracy: 0.9087 - val_loss: 0.2491
Epoch 6/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9251 - loss: 0.2039 - val_accuracy: 0.9110 - val_loss: 0.2439
Epoch 7/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9323 - loss: 0.1829 - val_accuracy: 0.9107 - val_loss: 0.2455
Epoch 8/15
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9398 - loss: 0.1651 - 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

yPredictProbability = model.predict(mXTest)
yPredict = np.argmax(yPredictProbability, axis=1)

confusionMatrix = confusion_matrix(mYTest, yPredict)
confusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix=confusionMatrix)
confusionMatrixDisplay.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

misclassifiedIndices = np.where(yPredict != mYTest)[0]

classNames = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

plt.figure(figsize=(10, 4))

for i in range(3):
    index = misclassifiedIndices[i]

    plt.subplot(1, 3, i + 1)
    plt.imshow(mXTest[index].reshape(28, 28), cmap='gray')
    plt.title(f"True: {classNames[mYTest[index]]}\nPredicted: {classNames[yPredict[index]]}")
    plt.axis('off')

plt.tight_layout()
plt.show()


# Applying augmentations to the data (i.e rotations, zoom) would improve the generalization, and reduce 
# the amount of misclassifications between similar classes

NameError: name 'model' is not defined